In [1]:
import pandas as pd
import numpy as np

### Load the new data file

In [2]:
file_path = r"C:\Users\HP\Desktop\Infotact Internship\ai4i_feature_engineered.csv"

In [3]:
df = pd.read_csv(file_path)

In [4]:
df.shape

(10000, 29)

In [5]:
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,...,Process temperature [K]_rolling_var,Rotational speed [rpm]_rolling_mean,Rotational speed [rpm]_rolling_std,Rotational speed [rpm]_rolling_var,Torque [Nm]_rolling_mean,Torque [Nm]_rolling_std,Torque [Nm]_rolling_var,Tool wear [min]_rolling_mean,Tool wear [min]_rolling_std,Tool wear [min]_rolling_var
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,...,0.018222,1521.6,113.060652,12782.711111,39.91,6.826655,46.603222,10.4,6.834553,46.711111
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,...,0.018222,1521.6,113.060652,12782.711111,39.91,6.826655,46.603222,10.4,6.834553,46.711111
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,...,0.018222,1521.6,113.060652,12782.711111,39.91,6.826655,46.603222,10.4,6.834553,46.711111
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,...,0.018222,1521.6,113.060652,12782.711111,39.91,6.826655,46.603222,10.4,6.834553,46.711111
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,...,0.018222,1521.6,113.060652,12782.711111,39.91,6.826655,46.603222,10.4,6.834553,46.711111


### Create Timestamp column

In [6]:
df['Timestamp'] = pd.date_range(start='2026-01-01', periods=len(df), freq='5min')

In [7]:
df[['Timestamp']].head()

,Timestamp
0,2026-01-01 00:00:00
1,2026-01-01 00:05:00
2,2026-01-01 00:10:00
3,2026-01-01 00:15:00
4,2026-01-01 00:20:00


### Create External context Dataset

In [8]:
np.random.seed(42)

In [9]:
context_df = pd.DataFrame({
    'Timestamp': df['Timestamp'],
    'ambient_temp': np.random.normal(loc=32, scale=3, size=len(df)),
    'humidity': np.random.randint(40,90,len(df)),
    'factory_load': np.random.randint(50,100,len(df))
    })

In [10]:
context_df.head()

,Timestamp,ambient_temp,humidity,factory_load
0,2026-01-01 00:00:00,33.490142,81,65
1,2026-01-01 00:05:00,31.585207,56,54
2,2026-01-01 00:10:00,33.943066,70,98
3,2026-01-01 00:15:00,36.569090,73,95
4,2026-01-01 00:20:00,31.297540,77,61


### Merge Internal and External Data

In [11]:
new_df = pd.merge(df, context_df, on='Timestamp', how='left')

In [12]:
new_df.shape

(10000, 33)

### Create Contextual Features

In [13]:
new_df['ambient_gap'] = (new_df['Process temperature [K]'] - new_df['ambient_temp'])

In [14]:
new_df['load_stress'] = (new_df['Torque [Nm]'] * new_df['factory_load'])

In [15]:
new_df['heat_stress'] = (new_df['ambient_temp'] * new_df['humidity'])

In [16]:
new_df['mechanical_stress'] = (new_df['Torque [Nm]'] * new_df['Rotational speed [rpm]'])

In [17]:
new_df['wear_efficiency'] = (new_df['Tool wear [min]'] / (new_df['Rotational speed [rpm]'] + 1))

In [18]:
new_df.shape

(10000, 38)

1) Ambient Gap - say about difference in machine temperature to the surrouding temperature.
2) Load Stress - represent how much mechanical load the machine experiences under operating conditions.
3) Heat Stress - represents how harsh the surrounding environmental conditions are for the machine.
4) Mechanical Stress - feature that estimates how hard the machine's moving parts are working.
5) Wear Efficiency - feature that relates the machine's wear to how much work it is doing.


### Encode Type Column

In [19]:
new_df['Type'].unique()

array(['M', 'L', 'H'], dtype=object)

In [20]:
new_df = pd.get_dummies(new_df, columns=['Type'], drop_first=True)

In [21]:
new_df.shape

(10000, 39)

### Correlation Study

In [22]:
corr = new_df.corr(numeric_only=True)

failure_corr = (
    corr['Machine failure']
    .sort_values(ascending=False)
)

print(failure_corr.head(20))

Machine failure                       1.000000
HDF                                   0.575800
OSF                                   0.531083
PWF                                   0.522812
TWF                                   0.362904
Torque [Nm]                           0.191321
mechanical_stress                     0.176039
load_stress                           0.148989
wear_efficiency                       0.130194
Torque [Nm]_rolling_var               0.110273
Tool wear [min]_rolling_mean          0.107343
Tool wear [min]                       0.105448
Torque [Nm]_rolling_std               0.105259
Air temperature [K]                   0.082556
Air temperature [K]_rolling_mean      0.082189
Rotational speed [rpm]_rolling_var    0.067868
Rotational speed [rpm]_rolling_std    0.062142
Torque [Nm]_rolling_mean              0.053765
Process temperature [K]               0.035946
Type_L                                0.035643
Name: Machine failure, dtype: float64


### Internal Features

In [26]:
internal_features = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]',

    'Air temperature [K]_rolling_mean',
    'Air temperature [K]_rolling_std',
    'Air temperature [K]_rolling_var',

    'Process temperature [K]_rolling_mean',
    'Process temperature [K]_rolling_std',
    'Process temperature [K]_rolling_var',

    'Rotational speed [rpm]_rolling_mean',
    'Rotational speed [rpm]_rolling_std',
    'Rotational speed [rpm]_rolling_var',

    'Torque [Nm]_rolling_mean',
    'Torque [Nm]_rolling_std',
    'Torque [Nm]_rolling_var',

    'Tool wear [min]_rolling_mean',
    'Tool wear [min]_rolling_std',
    'Tool wear [min]_rolling_var'
]

### Internal + Context Features

In [27]:
context_features = internal_features + [
    'ambient_temp',
    'humidity',
    'factory_load',
    'ambient_gap',
    'load_stress',
    'heat_stress'
]

In [28]:
print("Internal Features :", len(internal_features))
print("Context Features :", len(context_features))

Internal Features : 20
Context Features : 26


### save the new dataset with external features

In [ ]:
new_df.to_csv(
    "week2_fused_dataset.csv",
    index=False
)

In [29]:
import os
print(os.getcwd())

c:\Users\HP\Desktop\PredictX\notebooks
